# Smart Surveillance — Autoencoder Training
**No Google Drive needed.** Run cells 1→2→3→4→5→6→7 in order.
At the end, `anomaly_model.h5` and `threshold.npy` auto-download to your PC.

In [ ]:
# CELL 1 — Install (pinned kaggle version) & authenticate
!pip install -q 'kaggle==1.5.16' tensorflow matplotlib

from google.colab import files
print('Upload kaggle.json:')
files.upload()

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle authenticated OK')

In [ ]:
# CELL 2 — Download ONLY 7 categories (not full 95 GB)
import os, subprocess, csv, io, zipfile

DATASET  = 'webadvisor/real-time-anomaly-detection-in-cctv-surveillance'
REQUIRED = ['Normal','Fighting','Robbery','RoadAccident','Stealing','Shooting','Burglary']
DEST     = '/content/dataset'
os.makedirs(DEST, exist_ok=True)

# Step 1: List all files (fast — no download)
print('Listing dataset files...')
result = subprocess.run(
    ['kaggle', 'datasets', 'files', DATASET, '--csv'],
    capture_output=True, text=True
)
all_files = list(csv.DictReader(io.StringIO(result.stdout)))
print(f'Total files in dataset: {len(all_files):,}')

# Step 2: Filter to our 7 categories only
def in_category(fname):
    top = str(fname).replace('\\', '/').split('/')[0]
    return any(cat in top for cat in REQUIRED)

selected = [f for f in all_files if in_category(f.get('name',''))]
print(f'Files in our 7 categories: {len(selected):,}')

# Show per-category breakdown
for cat in REQUIRED:
    n = sum(1 for f in selected if cat in str(f.get('name','')).split('/')[0])
    print(f'  {cat}: {n} files')

# Step 3: Download each selected file
print('\nDownloading...')
for i, row in enumerate(selected, 1):
    fname   = row['name']
    out_dir = os.path.join(DEST, os.path.dirname(fname))
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(DEST, fname)
    if os.path.exists(out_file):
        continue
    subprocess.run(
        ['kaggle', 'datasets', 'download', DATASET,
         '--file', fname, '-p', out_dir, '--unzip'],
        capture_output=True
    )
    if i % 20 == 0 or i == len(selected):
        print(f'  {i}/{len(selected)} done')

# Step 4: Count extracted frames
print('\nFrame counts per category:')
for cat in REQUIRED:
    cnt = sum(
        len([x for x in fs if x.endswith(('.jpg','.png'))])
        for r, _, fs in os.walk(os.path.join(DEST, cat))
    )
    print(f'  {cat}: {cnt:,} frames')
print('Done!')

In [ ]:
# CELL 3 — Load Normal frames for training
import numpy as np, cv2, random
from pathlib import Path

IMG_SIZE = 128

def load_frames(folder, max_frames=50000, label=''):
    paths = list(Path(folder).rglob('*.jpg')) + list(Path(folder).rglob('*.png'))
    random.shuffle(paths)
    frames = []
    for p in paths[:max_frames]:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        frames.append(cv2.resize(img,(IMG_SIZE,IMG_SIZE)).astype('float32')/255.0)
    arr = np.array(frames).reshape(-1,IMG_SIZE,IMG_SIZE,1)
    print(f'  {label or folder}: {len(arr):,} frames')
    return arr

# Find Normal dir
normal_path = None
for root,_,fs in os.walk('/content/dataset'):
    if 'Normal' in root and any(f.endswith(('.jpg','.png')) for f in fs):
        normal_path = root; break

if not normal_path:
    raise FileNotFoundError('Normal folder not found')

print('Loading Normal frames...')
X_train = load_frames(normal_path, label='Normal')
print(f'Shape: {X_train.shape}  RAM: {X_train.nbytes/1e9:.2f} GB')

In [ ]:
# CELL 4 — Load anomaly frames for calibration (max 2000 each)
ANOMALY_NAMES = ['Fighting','Robbery','RoadAccident','Stealing','Shooting','Burglary']
anomaly_frames = {}

for name in ANOMALY_NAMES:
    path = None
    for root,_,fs in os.walk('/content/dataset'):
        if name in root and any(f.endswith(('.jpg','.png')) for f in fs):
            path = root; break
    if path:
        fr = load_frames(path, max_frames=2000, label=name)
        if len(fr): anomaly_frames[name] = fr
    else:
        print(f'  {name}: not found')

print(f'Loaded {len(anomaly_frames)} anomaly categories')

In [ ]:
# CELL 5 — Build Autoencoder
import tensorflow as tf
from tensorflow.keras import layers, Model
print(f'TF {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')

inp = tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,1))
x = layers.Conv2D(32,3,activation='relu',padding='same')(inp)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64,3,activation='relu',padding='same')(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128,3,activation='relu',padding='same')(x)
enc = layers.MaxPooling2D(2)(x)
x = layers.Conv2DTranspose(128,3,activation='relu',padding='same')(enc)
x = layers.UpSampling2D(2)(x)
x = layers.Conv2DTranspose(64,3,activation='relu',padding='same')(x)
x = layers.UpSampling2D(2)(x)
x = layers.Conv2DTranspose(32,3,activation='relu',padding='same')(x)
x = layers.UpSampling2D(2)(x)
out = layers.Conv2D(1,3,activation='sigmoid',padding='same')(x)

autoencoder = Model(inp, out, name='surveillance_ae')
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()
print(f'Parameters: {autoencoder.count_params():,}')

In [ ]:
# CELL 6 — Train (~30-60 min on T4)
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

SAVE_DIR   = '/content/surveillance_model'
MODEL_PATH = f'{SAVE_DIR}/anomaly_model.h5'
os.makedirs(SAVE_DIR, exist_ok=True)

history = autoencoder.fit(
    X_train, X_train, epochs=50, batch_size=32,
    validation_split=0.1, shuffle=True, verbose=1,
    callbacks=[
        ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]
)

plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val', linestyle='--')
plt.title('Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/loss_curve.png')
plt.show()
print(f'Best val_loss: {min(history.history["val_loss"]):.6f}')

In [ ]:
# CELL 7 — Calibrate threshold + auto-download to your PC
import json

autoencoder = tf.keras.models.load_model(MODEL_PATH, compile=False)

def get_errors(frames, model):
    preds = model.predict(frames, batch_size=64, verbose=0)
    return np.mean(np.power(frames-preds,2), axis=(1,2,3))

normal_errors = get_errors(X_train[-5000:], autoencoder)
threshold = float(np.percentile(normal_errors, 95))

print(f'Normal mean error : {np.mean(normal_errors):.6f}')
print(f'Threshold (95th%) : {threshold:.6f}')
print()

for name, frames in anomaly_frames.items():
    errs = get_errors(frames, autoencoder)
    print(f'{name:<16} mean={np.mean(errs):.5f}  p95={np.percentile(errs,95):.5f}')

THRESHOLD_PATH = f'{SAVE_DIR}/threshold.npy'
np.save(THRESHOLD_PATH, threshold)
json.dump({'threshold':threshold,'normal_mean':float(np.mean(normal_errors))},
          open(f'{SAVE_DIR}/model_report.json','w'), indent=2)

print('\nDownloading files to your PC...')
files.download(MODEL_PATH)
files.download(THRESHOLD_PATH)

print('DONE! Place both files in backend/models/ then restart Flask.')